<a href="https://colab.research.google.com/github/Gabalecrim/IAAplicada/blob/main/busca_em_arvore_PG_Antonina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Busca em Árvore — Portal da Graciosa → Antonina

Implementação executável do exercício de *Introdução à Busca* (Inteligência Artificial — Prof. Renato Dorighello).

**Problema.** Planejar a ida de **Portal da Graciosa (PG)** a **Antonina (A)** pelo caminho de menor custo. O espaço de estados é um **grafo dirigido acíclico (DAG)**.

**Estratégia da busca (enunciado, item b):**
- seleciona-se sempre o **nó mais raso** da fronteira (busca em largura / *breadth-first*);
- empate → escolhe a cidade por **ordem alfabética DECRESCENTE**;
- persistindo o empate → escolhe o nó **inserido antes** na fronteira (FIFO);
- **teste de objetivo tardio**: o objetivo só é testado quando o nó é **selecionado para expansão** (como na `TREE-SEARCH` clássica de Russell & Norvig).

## 1. Formulação do problema

Estados, estado inicial/objetivo e função sucessora com os **dois** custos possíveis (distância em km e tempo em minutos). Numa *busca de custo uniforme* usaríamos apenas **uma** medida; aqui a estratégia é por profundidade, então os custos servem só para **reportar** o custo do caminho encontrado.

In [ ]:
# Espaço de estados: grafo dirigido acíclico. Aresta = (vizinho, km, minutos)
GRAFO = {
    'PG': [('SJ', 19.1, 18)],
    'SJ': [('B', 17.7, 18), ('M', 13.2, 14)],
    'M':  [('B',  7.0,  8)],
    'B':  [('A',  6.9,  8)],
    'A':  [],
}
NOMES = {'PG': 'Portal da Graciosa', 'SJ': 'São João', 'M': 'Morretes',
         'B': 'Bufara', 'A': 'Antonina'}
INICIAL, OBJETIVO = 'PG', 'A'

print('Estados S =', set(GRAFO))
print('Estado inicial:', INICIAL, '(', NOMES[INICIAL], ')')
print('Estado objetivo:', OBJETIVO, '(', NOMES[OBJETIVO], ')\n')
print('Função sucessora:')
for s, arestas in GRAFO.items():
    for (v, km, mn) in arestas:
        print(f'  suc({s}, ir({v})) = {v:<3} | {km:>4} km | {mn:>2} min')

Estados S = {'M', 'A', 'B', 'PG', 'SJ'}
Estado inicial: PG ( Portal da Graciosa )
Estado objetivo: A ( Antonina )

Função sucessora:
  suc(PG, ir(SJ)) = SJ  | 19.1 km | 18 min
  suc(SJ, ir(B)) = B   | 17.7 km | 18 min
  suc(SJ, ir(M)) = M   | 13.2 km | 14 min
  suc(M, ir(B)) = B   |  7.0 km |  8 min
  suc(B, ir(A)) = A   |  6.9 km |  8 min


## 2. Nó da árvore e algoritmo `busca_em_arvore`

In [ ]:
import itertools

class No:
    """Nó da árvore de busca: estado, pai, ação, profundidade,
    custo acumulado (km e min) e a ordem de inserção na fronteira."""
    _contador = itertools.count()
    def __init__(self, estado, pai=None, acao=None, custo_km=0.0, custo_min=0.0):
        self.estado = estado
        self.pai = pai
        self.acao = acao
        self.profundidade = 0 if pai is None else pai.profundidade + 1
        self.custo_km = custo_km
        self.custo_min = custo_min
        self.ordem_insercao = next(No._contador)
        self.filhos = []                       # usado apenas para desenhar a árvore
    def caminho(self):
        no, seq = self, []
        while no is not None:
            seq.append(no.estado); no = no.pai
        return list(reversed(seq))

def _chave(no, desempate):
    """Chave de seleção: (1) nó mais RASO; (2) desempate por nome de cidade
    (DECRESCENTE do enunciado, ou CRESCENTE); (3) quem entrou antes na fronteira (FIFO)."""
    if desempate == 'decrescente':
        nome = tuple(-ord(c) for c in no.estado)   # nome "maior" => escolhido primeiro
    else:
        nome = no.estado                           # nome "menor" => escolhido primeiro
    return (no.profundidade, nome, no.ordem_insercao)

def busca_em_arvore(desempate='decrescente'):
    """TREE-SEARCH: sem lista de explorados; teste de objetivo só na seleção para expansão."""
    No._contador = itertools.count()
    raiz = No(INICIAL)
    fronteira = [raiz]
    gerados   = [raiz]          # todo nó criado = 1 vértice da árvore de busca
    log       = []
    while fronteira:
        no = min(fronteira, key=lambda n: _chave(n, desempate))   # nó mais raso + desempates
        fronteira.remove(no)
        objetivo = (no.estado == OBJETIVO)
        gerou = []
        if not objetivo:                                          # expande o nó
            for (viz, km, mn) in GRAFO[no.estado]:
                filho = No(viz, no, f'ir({viz})', no.custo_km + km, no.custo_min + mn)
                no.filhos.append(filho); fronteira.append(filho); gerados.append(filho)
                gerou.append(filho.estado)
        log.append({'passo': len(log)+1,
                    'escolhido': f'{no.estado}(d{no.profundidade})',
                    'objetivo': objetivo,
                    'gerou': gerou,
                    'fronteira': [f'{n.estado}(d{n.profundidade})' for n in fronteira]})
        if objetivo:
            return {'solucao': no, 'gerados': gerados, 'log': log, 'raiz': raiz}
    return {'solucao': None, 'gerados': gerados, 'log': log, 'raiz': raiz}

print('Algoritmo definido.')

Algoritmo definido.


## 3. Execução e traço da busca

In [ ]:
res = busca_em_arvore(desempate='decrescente')

print('TRAÇO (mais raso primeiro; desempate alfabético DECRESCENTE; objetivo testado na expansão)\n')
print(f"{'passo':>5} | {'expande':>8} | objetivo | {'gerou':<9} | fronteira após o passo")
print('-' * 78)
for e in res['log']:
    print(f"{e['passo']:>5} | {e['escolhido']:>8} | {('SIM' if e['objetivo'] else 'não'):^8} | "
          f"{(','.join(e['gerou']) or '-'):<9} | {', '.join(e['fronteira']) or '(vazia)'}")

sol = res['solucao']
print('\nSOLUÇÃO encontrada:')
print('  caminho  :', ' → '.join(sol.caminho()))
print(f'  profund. : d = {sol.profundidade}')
print(f'  custo    : {sol.custo_km:.1f} km  |  {int(sol.custo_min)} min')

TRAÇO (mais raso primeiro; desempate alfabético DECRESCENTE; objetivo testado na expansão)

passo |  expande | objetivo | gerou     | fronteira após o passo
------------------------------------------------------------------------------
    1 |   PG(d0) |   não    | SJ        | SJ(d1)
    2 |   SJ(d1) |   não    | B,M       | B(d2), M(d2)
    3 |    M(d2) |   não    | B         | B(d2), B(d3)
    4 |    B(d2) |   não    | A         | B(d3), A(d3)
    5 |    B(d3) |   não    | A         | A(d3), A(d4)
    6 |    A(d3) |   SIM    | -         | A(d4)

SOLUÇÃO encontrada:
  caminho  : PG → SJ → B → A
  profund. : d = 3
  custo    : 43.7 km  |  44 min


## 4. Árvore de busca gerada

In [ ]:
def desenhar_arvore(no, prefixo='', conector=''):
    marca = '   ← OBJETIVO' if no.estado == OBJETIVO else ''
    print(prefixo + conector + f'{no.estado} (d{no.profundidade})' + marca)
    if   conector == '└── ': novo = prefixo + '    '
    elif conector == '├── ': novo = prefixo + '│   '
    else:                       novo = prefixo
    n = len(no.filhos)
    for i, filho in enumerate(no.filhos):
        desenhar_arvore(filho, novo, '└── ' if i == n-1 else '├── ')

print('ÁRVORE DE BUSCA (desempate DECRESCENTE):\n')
desenhar_arvore(res['raiz'])
print('\nNota: Bufara (B) aparece 2× (via SJ e via M) e Antonina (A) aparece nos ramos expandidos.')

ÁRVORE DE BUSCA (desempate DECRESCENTE):

PG (d0)
└── SJ (d1)
    ├── B (d2)
    │   └── A (d3)   ← OBJETIVO
    └── M (d2)
        └── B (d3)
            └── A (d4)   ← OBJETIVO

Nota: Bufara (B) aparece 2× (via SJ e via M) e Antonina (A) aparece nos ramos expandidos.


## 5. Medidas de complexidade (item c)

Comparação **espaço de estados × árvore de busca**:
1. número de vértices;
2. tamanho máximo de caminho no grafo × profundidade máxima da árvore (**m**);
3. profundidade do objetivo mais raso (**d**) × menor caminho (em arestas) do inicial ao objetivo;
4. fator de ramificação máximo (**b**).

In [ ]:
from collections import deque

def caminho_mais_longo(grafo):
    """Maior caminho simples (em arestas) no DAG — memória via DFS."""
    memo = {}
    def dfs(u):
        if u in memo: return memo[u]
        melhor = 0
        for (v, _, _) in grafo[u]:
            melhor = max(melhor, 1 + dfs(v))
        memo[u] = melhor
        return melhor
    return max(dfs(u) for u in grafo)

def menor_caminho_arestas(grafo, ini, obj):
    """Menor caminho em número de arestas (BFS não ponderado)."""
    fila, visto = deque([(ini, 0)]), {ini}
    while fila:
        u, d = fila.popleft()
        if u == obj: return d
        for (v, _, _) in grafo[u]:
            if v not in visto:
                visto.add(v); fila.append((v, d+1))
    return None

gerados = res['gerados']
tabela = [
    ('nº de vértices',                         len(GRAFO),                               len(gerados)),
    ('tam. máx. caminho  /  prof. máx (m)',      caminho_mais_longo(GRAFO),                max(n.profundidade for n in gerados)),
    ('prof. objetivo mais raso (d)',            menor_caminho_arestas(GRAFO, INICIAL, OBJETIVO), res['solucao'].profundidade),
    ('fator de ramificação máx. (b)',            max(len(v) for v in GRAFO.values()),      max((len(n.filhos) for n in gerados), default=0)),
]

print(f"{'medida':<38} | {'espaço de estados':>17} | {'árvore de busca':>16}")
print('-' * 78)
for nome, a, b in tabela:
    print(f'{nome:<38} | {a:>17} | {b:>16}')

medida                                 | espaço de estados |  árvore de busca
------------------------------------------------------------------------------
nº de vértices                         |                 5 |                7
tam. máx. caminho  /  prof. máx (m)    |                 4 |                4
prof. objetivo mais raso (d)           |                 3 |                3
fator de ramificação máx. (b)          |                 2 |                2


## 6. Observação sobre a contagem de nós (DECRESCENTE × CRESCENTE)

Seguindo **à risca** o desempate **DECRESCENTE** do enunciado, a árvore tem **7 nós**: ao chegar à profundidade 3, a regra manda expandir **B** (B > A) *antes* de selecionar o objetivo **A**, e essa expansão gera um segundo **A** (ramo redundante `PG-SJ-M-B-A`, profundidade 4). É isso que torna **m = 4**, coerente com o item (c.ii).

O gabarito conta **6 nós** (Bufara 2×, Antonina 1×). Esse número aparece quando o objetivo **A** é selecionado *antes* de o segundo **B** ser expandido — ou seja, com desempate **CRESCENTE** (A < B). Aí a profundidade máxima efetivamente atingida é 3, não 4.

Ou seja, há uma pequena inconsistência interna no gabarito: *6 nós* combina com profundidade 3, enquanto *m = 4* exige o 7º nó. **O caminho-solução `PG → SJ → B → A` (d = 3, 43,7 km, 44 min) é o mesmo nas duas leituras.** A célula abaixo mostra os dois cenários lado a lado.

In [ ]:
for modo in ('decrescente', 'crescente'):
    r = busca_em_arvore(desempate=modo)
    ger = r['gerados']
    print(f'desempate {modo:>12}: '
          f'nós na árvore = {len(ger)} '
          f'{[n.estado for n in ger]}, '
          f'prof. máx = {max(n.profundidade for n in ger)}, '
          f"caminho = {' → '.join(r['solucao'].caminho())}")

desempate  decrescente: nós na árvore = 7 ['PG', 'SJ', 'B', 'M', 'B', 'A', 'A'], prof. máx = 4, caminho = PG → SJ → B → A
desempate    crescente: nós na árvore = 6 ['PG', 'SJ', 'B', 'M', 'A', 'B'], prof. máx = 3, caminho = PG → SJ → B → A
